# 🧪 Prefilling Practice — Different Output Formats

This notebook demonstrates the **prefilling** technique to control Claude's output format.

| Example | Prefill | Save as |
|---------|---------|----------|
| 1 | `` ```python `` | `.py` file |
| 2 | `#` | `.md` Markdown |
| 3 | `1.` | `.txt` plain text |
| 4 | `const dataset = [` | `.js` JavaScript |

In [21]:
# Setup: load environment variables
from dotenv import load_dotenv

load_dotenv(override=True)

True

In [22]:
# Setup: create API client
from anthropic import Anthropic

client = Anthropic()
model = "claude-haiku-4-5"

In [23]:
# Setup: helper functions
def add_user_message(messages, text):
    messages.append({"role": "user", "content": text})


def add_assistant_message(messages, text):
    messages.append({"role": "assistant", "content": text})


def chat(messages, system=None, temperature=1.0, stop_sequences=[]):
    params = {
        "model": model,
        "max_tokens": 800,
        "messages": messages,
        "temperature": temperature,
    }

    if system:
        params["system"] = system

    # Only add stop_sequences if the list is non-empty — API rejects empty arrays
    if stop_sequences:
        params["stop_sequences"] = stop_sequences
    message = client.messages.create(**params)
    return message.content[0].text

---
## 🐍 Example 1: Prefill with ` ```python ` → save as `.py`

We prefill Claude's response with ` ```python ` so it immediately writes Python code without any intro text.
The stop sequence ` ``` ` halts generation before the closing backticks.

In [24]:
def generate_as_python():
    prompt = """
    Write a single Python function that lists all S3 buckets using boto3.
    Return only the function, no explanations.
    """
    messages = []
    add_user_message(messages, prompt)

    # Prefill: we insert the start of the python code block
    # Claude "thinks" it already started writing code and continues without any intro
    add_assistant_message(messages, "```python")

    # Stop sequence: stop before the closing ```
    text = chat(messages, stop_sequences=["```"])
    return text


python_code = generate_as_python()
print("=== Result (clean Python) ===")
print(python_code)

=== Result (clean Python) ===

import boto3

def list_s3_buckets():
    """List all S3 buckets in the AWS account."""
    s3_client = boto3.client('s3')
    response = s3_client.list_buckets()
    buckets = [bucket['Name'] for bucket in response['Buckets']]
    return buckets



In [25]:
# Save as .py file
with open("generated_code.py", "w") as f:
    f.write(python_code.strip())

print("Saved to generated_code.py ✓")

Saved to generated_code.py ✓


---
## 📝 Example 2: Prefill with `#` → save as `.md` (Markdown)

We prefill with `#` — Claude immediately starts writing a Markdown heading.

> ⚠️ Note: the prefill must NOT end with a trailing space — the API will reject it.
> Use `"#"` (no space), then prepend `"# "` to the returned text.

In [26]:
def generate_as_markdown():
    prompt = """
    Create a short AWS S3 cheat sheet with the 3 most important AWS CLI commands for S3.
    Format it as Markdown with a title, brief description for each command, and code examples.
    """
    messages = []
    add_user_message(messages, prompt)

    # Prefill: insert "#" — the Markdown heading symbol
    # Claude immediately writes "# AWS S3 Cheat Sheet" and continues
    # IMPORTANT: no trailing space — API rejects prefills ending with whitespace
    add_assistant_message(messages, "#")

    # No stop sequence — let Claude write the full response
    text = chat(messages)
    return "# " + text  # restore the prefill (not included in the response)


markdown_content = generate_as_markdown()
print("=== Result (Markdown) ===")
print(markdown_content)

=== Result (Markdown) ===
# AWS S3 CLI Cheat Sheet

## 1. List S3 Buckets and Contents
**Description:** View your S3 buckets or list objects within a specific bucket.

```bash
# List all buckets
aws s3 ls

# List contents of a bucket
aws s3 ls s3://your-bucket-name/

# List with detailed info (size, date)
aws s3 ls s3://your-bucket-name/ --recursive --human-readable --summarize
```

## 2. Copy Files Between Local and S3
**Description:** Upload files to S3 or download files from S3.

```bash
# Upload a file to S3
aws s3 cp myfile.txt s3://your-bucket-name/

# Download a file from S3
aws s3 cp s3://your-bucket-name/myfile.txt ./myfile.txt

# Sync entire directory (upload)
aws s3 sync ./local-folder s3://your-bucket-name/folder/

# Sync from S3 (download)
aws s3 sync s3://your-bucket-name/folder/ ./local-folder
```

## 3. Remove Objects from S3
**Description:** Delete files or entire buckets from S3.

```bash
# Delete a single object
aws s3 rm s3://your-bucket-name/myfile.txt

# Delete al

In [27]:
# Save as .md file
with open("aws_cheatsheet.md", "w") as f:
    f.write(markdown_content)

print("Saved to aws_cheatsheet.md ✓")

Saved to aws_cheatsheet.md ✓


---
## 📄 Example 3: Prefill with `1.` → save as `.txt`

We prefill with `1.` — Claude immediately starts a numbered list without any intro like "Sure! Here are...".

In [28]:
def generate_as_text():
    prompt = """
    List 5 common AWS IAM best practices.
    Write them as a simple numbered list, one sentence each.
    """
    messages = []
    add_user_message(messages, prompt)

    # Prefill: insert "1." — Claude immediately writes the list
    # without "Sure! Here are 5 best practices:"
    add_assistant_message(messages, "1.")

    text = chat(messages)
    return "1." + text  # restore the prefill


plain_text = generate_as_text()
print("=== Result (Plain Text) ===")
print(plain_text)

=== Result (Plain Text) ===
1.  Enable multi-factor authentication (MFA) for all users, especially those with administrative privileges.

2. Create and use IAM roles instead of access keys for applications and services running on AWS resources.

3. Apply the principle of least privilege by granting users only the permissions they need to perform their job.

4. Regularly audit and remove unused IAM users, roles, and access keys to reduce security risks.

5. Use IAM policy conditions to restrict access based on IP addresses, time of day, or other contextual factors.


In [29]:
# Save as .txt file
with open("iam_best_practices.txt", "w") as f:
    f.write(plain_text)

print("Saved to iam_best_practices.txt ✓")

Saved to iam_best_practices.txt ✓


---
## 🟨 Example 4: Prefill with `const dataset = [` → save as `.js`

We prefill with the start of a JS variable — Claude generates a valid JavaScript array.
The stop sequence `];` halts generation after the array closes.

In [30]:
def generate_as_javascript():
    prompt = """
    Generate 3 AWS task objects for a JavaScript dataset.
    Each object should have: id (number), task (string), type ("python"|"json"|"regex").
    Output only the array contents (objects), no outer brackets.
    """
    messages = []
    add_user_message(messages, prompt)

    # Prefill: insert the start of a JS constant
    # Claude "thinks" it already started writing JS and continues from here
    add_assistant_message(messages, "const dataset = [")

    # Stop sequence: stop after the closing ];
    text = chat(messages, stop_sequences=["];\n", "];"])
    return "const dataset = [" + text + "];"  # assemble the full JS


js_code = generate_as_javascript()
print("=== Result (JavaScript) ===")
print(js_code)

=== Result (JavaScript) ===
const dataset = [
  { id: 1, task: "Parse S3 bucket logs and extract error codes", type: "python" },
  { id: 2, task: "Validate CloudFormation template structure", type: "json" },
  { id: 3, task: "Extract EC2 instance IDs from CloudWatch logs", type: "regex" }
];


In [31]:
# Save as .js file
with open("dataset.js", "w") as f:
    f.write(js_code)

print("Saved to dataset.js ✓")

Saved to dataset.js ✓


---
## 📊 Summary: Prefilling Technique

| Desired output | Prefill | Stop sequence | Save as |
|----------------|---------|---------------|---------|
| JSON array | ` ```json ` | ` ``` ` | `.json` |
| Python code | ` ```python ` | ` ``` ` | `.py` |
| Markdown | `#` | *(none)* | `.md` |
| Numbered list | `1.` | *(none)* | `.txt` |
| JavaScript | `const x = [` | `];` | `.js` |

> **Rule**: Prefill = the first characters of the format you expect to receive.

> ⚠️ **Gotcha**: Prefill content must NOT end with trailing whitespace — the API will return a 400 error.